In [1]:
import sys
sys.path.append("../src")

from data_loader import load_raw_data, time_based_split
from features import add_velocity_features, add_amount_deviation_features

data = load_raw_data()
train_df, test_df = time_based_split(data)
del data

Train range: 86400 to 12192842
Test range:  12192900 to 15811131
Train shape: (472432, 434), Test shape: (118108, 434)
Train fraud rate: 3.51%
Test fraud rate:  3.44%


In [2]:
train_df = add_velocity_features(train_df)
train_df = add_amount_deviation_features(train_df)

train_df[["TransactionID", "card1", "TransactionAmt", 
          "card1_txn_count_1d", "card1_txn_amt_sum_1d",
          "amt_deviation_from_avg"]].head(20)

,TransactionID,card1,TransactionAmt,card1_txn_count_1d,card1_txn_amt_sum_1d,amt_deviation_from_avg
0,2987000,13926,68.500000,0.0,0.0,0.0
1,2987001,2755,29.000000,0.0,0.0,0.0
2,2987002,4663,59.000000,0.0,0.0,0.0
3,2987003,18132,50.000000,0.0,0.0,0.0
4,2987004,4497,50.000000,0.0,0.0,0.0
5,2987005,5937,49.000000,0.0,0.0,0.0
6,2987006,12308,159.000000,0.0,0.0,0.0
7,2987007,12695,422.500000,0.0,0.0,0.0
8,2987008,2803,15.000000,0.0,0.0,0.0
9,2987009,17399,117.000000,0.0,0.0,0.0


In [3]:
train_df[train_df["card1_txn_count_1d"] > 0][
    ["TransactionID", "card1", "TransactionAmt", "card1_txn_count_1d", 
     "card1_txn_amt_sum_1d", "amt_deviation_from_avg"]
].head(10)

,TransactionID,card1,TransactionAmt,card1_txn_count_1d,card1_txn_amt_sum_1d,amt_deviation_from_avg
18,2987018,4663,47.950001,1.0,59.000000,0.000000
51,2987051,7835,226.000000,1.0,200.000000,0.000000
57,2987057,11839,50.000000,1.0,10.500000,0.000000
61,2987061,12544,58.950001,1.0,49.000000,0.000000
62,2987062,18132,200.000000,1.0,50.000000,0.000000
69,2987069,12866,20.000000,1.0,40.000000,0.000000
73,2987073,1955,554.000000,1.0,500.000000,0.000000
74,2987074,15885,27.792999,1.0,42.293999,0.000000
75,2987075,4806,68.500000,1.0,77.000000,0.000000
76,2987076,18132,36.950001,2.0,250.000000,-0.830143


In [5]:
from features import add_entity_risk_features

train_df = add_entity_risk_features(train_df, entity_col="P_emaildomain")

train_df[["TransactionID", "P_emaildomain", "isFraud", "P_emaildomain_fraud_rate"]].head(20)

,TransactionID,P_emaildomain,isFraud,P_emaildomain_fraud_rate
0,2987000,NaN,0,0.035135
1,2987001,gmail.com,0,0.035135
2,2987002,outlook.com,0,0.035135
3,2987003,yahoo.com,0,0.035135
4,2987004,gmail.com,0,0.000000
5,2987005,gmail.com,0,0.000000
6,2987006,yahoo.com,0,0.000000
7,2987007,mail.com,0,0.035135
8,2987008,anonymous.com,0,0.035135
9,2987009,yahoo.com,0,0.000000


In [6]:
#Step 1: Apply entity risk to card1 too
train_df = add_entity_risk_features(train_df, entity_col="card1")

train_df[["TransactionID", "card1", "isFraud", "card1_fraud_rate"]].head(20)


,TransactionID,card1,isFraud,card1_fraud_rate
0,2987000,13926,0,0.035135
1,2987001,2755,0,0.035135
2,2987002,4663,0,0.035135
3,2987003,18132,0,0.035135
4,2987004,4497,0,0.035135
5,2987005,5937,0,0.035135
6,2987006,12308,0,0.035135
7,2987007,12695,0,0.035135
8,2987008,2803,0,0.035135
9,2987009,17399,0,0.035135


In [7]:
from features import encode_categoricals

train_df = encode_categoricals(train_df)
train_df.dtypes.value_counts()

Encoded 31 categorical columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


float32     399
category     12
float64       9
int32         4
category      4
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64

In [8]:
test_df = add_velocity_features(test_df)
test_df = add_amount_deviation_features(test_df)
test_df = add_entity_risk_features(test_df, entity_col="P_emaildomain")
test_df = add_entity_risk_features(test_df, entity_col="card1")
test_df = encode_categoricals(test_df)

print(test_df.shape)

Encoded 31 categorical columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
(118108, 443)


In [9]:
train_df.to_parquet("../data/processed/train_features.parquet")
test_df.to_parquet("../data/processed/test_features.parquet")
print("Saved processed data")

Saved processed data


In [10]:
print("Total columns:", len(test_df.columns))
print("\nEngineered feature columns:")
engineered = [c for c in test_df.columns if any(x in c for x in 
              ["txn_count", "txn_amt_sum", "expanding", "deviation", "fraud_rate"])]
print(engineered)

Total columns: 443

Engineered feature columns:
['card1_txn_count_1d', 'card1_txn_amt_sum_1d', 'card1_txn_count_7d', 'card1_txn_amt_sum_7d', 'card1_amt_expanding_mean', 'card1_amt_expanding_std', 'amt_deviation_from_avg', 'P_emaildomain_fraud_rate', 'card1_fraud_rate']
